<a href="https://colab.research.google.com/github/shravan1808/DeepLearningProjects/blob/main/Project_02/Project_02_DL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
X_raw = np.array([
    [0.15,  2, 0, 1], [0.85, 35, 4, 3], [0.95, 80, 8, 4], [0.10,  1, 0, 1], [0.78, 28, 3, 3],
    [0.98, 92, 9, 4], [0.20,  3, 0, 1], [0.82, 30, 4, 3], [0.92, 75, 7, 4], [0.18,  2, 0, 1],
    [0.88, 33, 3, 3], [0.99, 88, 8, 4], [0.12,  1, 0, 1], [0.80, 29, 3, 3], [0.96, 82, 8, 4],
    [0.22,  4, 0, 1], [0.84, 32, 4, 3], [0.91, 78, 7, 4], [0.19,  2, 0, 1], [0.86, 34, 3, 3],
    [0.94, 85, 8, 4], [0.14,  1, 0, 1], [0.81, 31, 3, 3], [0.97, 90, 9, 4], [0.25,  5, 0, 1],
    [0.83, 33, 4, 3], [0.93, 81, 7, 4], [0.16,  2, 0, 1], [0.87, 36, 3, 3], [0.90, 76, 7, 4],
    [0.21,  3, 0, 1], [0.79, 27, 3, 3], [0.98, 89, 8, 4], [0.11,  1, 0, 1], [0.85, 35, 4, 3],
    [0.95, 84, 8, 4], [0.23,  4, 0, 1], [0.82, 32, 3, 3], [0.92, 79, 7, 4], [0.17,  2, 0, 1],
    [0.89, 37, 4, 3], [0.99, 91, 9, 4], [0.13,  1, 0, 1], [0.80, 30, 3, 3], [0.94, 83, 8, 4],
    [0.24,  4, 0, 1], [0.86, 34, 4, 3], [0.96, 86, 8, 4], [0.18,  2, 0, 1], [0.88, 35, 3, 3],
    [0.91, 77, 7, 4], [0.15,  1, 0, 1], [0.83, 31, 3, 3], [0.97, 88, 8, 4], [0.20,  3, 0, 1],
    [0.85, 33, 4, 3], [0.93, 80, 7, 4], [0.12,  1, 0, 1], [0.81, 29, 3, 3], [0.95, 85, 8, 4]
])

In [ ]:
X_mean = X_raw.mean(axis=0)
X_std = X_raw.std(axis=0) + 1e-8
X_data = (X_raw - X_mean) / X_std

In [ ]:
y_raw = np.array([
    0, 1, 2, 0, 1, 2, 0, 1, 2, 0,
    1, 2, 0, 1, 2, 0, 1, 2, 0, 1,
    2, 0, 1, 2, 0, 1, 2, 0, 1, 2,
    0, 1, 2, 0, 1, 2, 0, 1, 2, 0,
    1, 2, 0, 1, 2, 0, 1, 2, 0, 1,
    2, 0, 1, 2, 0, 1, 2, 0, 1, 2
])

In [ ]:
num_classes = 3
y_data = np.zeros((y_raw.size, num_classes))
y_data[np.arange(y_raw.size), y_raw] = 1.0

In [ ]:
def relu(z):
  return np.maximum(0,z)

def relu_derivative(z):
  return (z>0).astype(float)

def softmax(z):
  Z_shifted = z - np.max(z, axis=1, keepdims=True)
  exp_z = np.exp(Z_shifted)
  return exp_z / np.sum(exp_z, axis=1, keepdims=True)

In [ ]:
z = np.array([[1.0, 2.0, 3.0]])
probs = softmax(z)
print("[TASK 1 SUCCESS] Multi-Class Activation Engine Verified:")
print(f"ReLU(-2.5) = {relu(-2.5)} | ReLU'(2.5) = {relu_derivative(np.array(2.5))}")
print(f"Softmax Probabilities Sum: {np.sum(probs):.4f} | Sample Output: {probs}")

[TASK 1 SUCCESS] Multi-Class Activation Engine Verified:
ReLU(-2.5) = 0.0 | ReLU'(2.5) = 1.0
Softmax Probabilities Sum: 1.0000 | Sample Output: [[0.09003057 0.24472847 0.66524096]]


In [ ]:
class DenseLayer:
  def __init__(self,input_size,output_size,activation,initialization):
    self.input_size = input_size
    self.output_size = output_size
    self.activation = activation
    if initialization == 'he':
      self.weights = np.random.randn(input_size, output_size) * np.sqrt(2/input_size)
    elif initialization == 'xavier':
      self.weights = np.random.randn(input_size, output_size) * np.sqrt(1/input_size)
    self.biases = np.zeros((1, output_size))
    self.velocity_W = np.zeros_like(self.weights)
    self.velocity_b = np.zeros_like(self.biases)
    self.input = None
    self.Z = None
    self.A = None
    self.dW = None
    self.db = None

  def forward(self,X):
    self.input = X
    self.Z = X @ self.weights + self.biases
    self.A = self.activation(self.Z)
    return self.A

  def backward(self,gradient,gradient_is_dZ= False):
    if gradient_is_dZ:
      dZ = gradient
    else:
      dZ = gradient * relu_derivative(self.Z)
    N = self.input.shape[0]
    self.dW = self.input.T @ dZ / N
    self.db = np.sum(dZ,axis = 0, keepdims = True) / N
    dA_prev = dZ @ self.weights.T
    return dA_prev

  def update_parameters(self,learning_rate,beta):
    self.velocity_W = beta * self.velocity_W + (1-beta) * self.dW
    self.velocity_b = beta * self.velocity_b + (1-beta) * self.db
    self.weights = self.weights - learning_rate * self.velocity_W
    self.biases = self.biases - learning_rate * self.velocity_b

In [ ]:
class NeuralNetwork:
  def __init__(self,layers):
    self.layers = layers

  def forward(self,X):
    A  = X
    for layer in self.layers:
      A = layer.forward(A)
    return A

  def compute_loss(self,A,y):
    N = y.shape[0]
    A_clipped = np.clip(A, 1e-15, 1-1e-15)
    loss = - np.sum(y * np.log(A_clipped))/N
    return loss

  def backward(self,y):
    output_layer = self.layers[-1]
    dZ = output_layer.A - y
    dA = output_layer.backward(dZ,gradient_is_dZ=True)
    for layer in reversed(self.layers[:-1]):
      dA= layer.backward(dA)

  def update_parameters(self,learning_rate,beta):
    for layer in self.layers:
      layer.update_parameters(learning_rate,beta)

  def train(self,X,y,epochs,learning_rate,beta):
    loss_history=[]
    for epoch in range(epochs):
      predictions = self.forward(X)
      loss = self.compute_loss(predictions,y)
      self.backward(y)
      self.update_parameters(learning_rate,beta)
      loss_history.append(loss)
    return loss_history

In [ ]:
np.random.seed(42)
layer1 = DenseLayer(4, 16, relu, "he")
layer2 = DenseLayer(16,8,relu,"he")
layer3 = DenseLayer(8,3,softmax,'xavier')

In [ ]:
old_weights = layer1.weights.copy()

In [ ]:
layers = [layer1,layer2,layer3]
network = NeuralNetwork(layers)
history = network.train(X_data,y_data,epochs = 120,learning_rate = 0.1,beta = 0.9)

In [ ]:
print("[TASK 2 SUCCESS] Dense Layer Initialized:")
print(f"Layer 1 (He Init) Weights Shape: {layer1.weights.shape} | Biases Shape: {layer1.biases.shape}")
print(f"Layer 3 (Xavier Init) Weights Shape: {layer3.weights.shape} | Biases Shape: {layer3.biases.shape}")

[TASK 2 SUCCESS] Dense Layer Initialized:
Layer 1 (He Init) Weights Shape: (4, 16) | Biases Shape: (1, 16)
Layer 3 (Xavier Init) Weights Shape: (8, 3) | Biases Shape: (1, 3)


In [ ]:
predictions = network.forward(X_data)
predicted_classes = np.argmax(predictions, axis=1)
class_dis = np.bincount(predicted_classes)

In [ ]:
print("[TASK 3 SUCCESS] Dynamic Forward Propagation Complete:")
print(f"Output Prediction Matrix Shape: {predictions.shape}")
print(f"Sample Class Probabilities (First 3): {np.round(predictions[0],3)}, {np.round(predictions[1],3)}, {np.round(predictions[2],3)}")

[TASK 3 SUCCESS] Dynamic Forward Propagation Complete:
Output Prediction Matrix Shape: (60, 3)
Sample Class Probabilities (First 3): [0.997 0.003 0.   ], [0.033 0.909 0.058], [0.002 0.004 0.995]


In [ ]:
print("[TASK 4 SUCCESS] Initial CCE Loss Engine Computed:")
print(f"Initial Un-optimized Loss: {history[0]:.4f}")

[TASK 4 SUCCESS] Initial CCE Loss Engine Computed:
Initial Un-optimized Loss: 1.3264


In [ ]:
print("[TASK 5 SUCCESS] Backpropagation Engine Execution:")
print(f"Output Layer Gradient dW Shape: {layer3.dW.shape} | Hidden Layer 1 Gradient dW Shape: {layer2.dW.shape}")
print(f"Layer 1 Gradient Norm: {np.linalg.norm(layer1.velocity_W):.4f} | Layer 3 Gradient Norm: {np.linalg.norm(layer3.velocity_W):.4f}")

[TASK 5 SUCCESS] Backpropagation Engine Execution:
Output Layer Gradient dW Shape: (8, 3) | Hidden Layer 1 Gradient dW Shape: (16, 8)
Layer 1 Gradient Norm: 0.0214 | Layer 3 Gradient Norm: 0.0633


In [ ]:
velocity_norm = np.linalg.norm(layer1.velocity_W)
weight_delta_norm = np.linalg.norm(layer1.weights - old_weights)
print("[TASK 6 SUCCESS] Momentum SGD Optimizer Applied:")
print(f"Layer 1 Velocity Norm: {velocity_norm:.4f}")
print(f"Layer 1 Weight Delta Norm: {weight_delta_norm:.4f}")
print("Velocity Integration Confirmed: True")

[TASK 6 SUCCESS] Momentum SGD Optimizer Applied:
Layer 1 Velocity Norm: 0.0214
Layer 1 Weight Delta Norm: 0.8477
Velocity Integration Confirmed: True


In [ ]:
print("[TASK 7 SUCCESS] Modular Neural Network Training Progress:")
for epoch in [1, 30, 60, 90, 120]:
    print(f"Epoch {epoch:03d} | CCE Loss: {history[epoch - 1]:.4f}")

[TASK 7 SUCCESS] Modular Neural Network Training Progress:
Epoch 001 | CCE Loss: 1.3264
Epoch 030 | CCE Loss: 0.4766
Epoch 060 | CCE Loss: 0.1828
Epoch 090 | CCE Loss: 0.0780
Epoch 120 | CCE Loss: 0.0330


In [ ]:
print("[TASK 8 SUCCESS] Argmax Classification Thresholding Applied:")
print(f"Discrete Class Predictions Shape: {predicted_classes.shape}")
print(f"Predicted Class Distribution -> Basic (0): {class_dis[0]} | Pro (1): {class_dis[1]} | Enterprise (2): {class_dis[2]}")

[TASK 8 SUCCESS] Argmax Classification Thresholding Applied:
Discrete Class Predictions Shape: (60,)
Predicted Class Distribution -> Basic (0): 20 | Pro (1): 20 | Enterprise (2): 20


In [ ]:
from sklearn.metrics import accuracy_score
true_classes = np.argmax(y_data, axis=1)
correct = np.sum(predicted_classes == true_classes)
accuracy = accuracy_score(true_classes,predicted_classes)
print(f"Accuracy: {accuracy * 100:.2f}%")

print("[TASK 9 SUCCESS] Categorical Accuracy Evaluated:")
print(f"Training Classification Accuracy: {accuracy*100:.2f}%")
print(f"Correct Classifications: {correct} / {len(true_classes)} Samples")

Accuracy: 100.00%
[TASK 9 SUCCESS] Categorical Accuracy Evaluated:
Training Classification Accuracy: 100.00%
Correct Classifications: 60 / 60 Samples


In [ ]:
new_customer = np.array([[0.95, 80, 8, 4]])
new_customer_normalized = (new_customer - X_mean)/X_std
new_customer_pred = network.forward(new_customer_normalized)
predicted_class = np.argmax(new_customer_pred,axis=1)[0]
print("[TASK 10 SUCCESS] Single-Sample Inference Execution:")
print(f"Raw Sample Input: {np.round(new_customer,2)}")
print(f"Normalized Vector: {np.round(new_customer_normalized,4)}")
print(f"Class Probabilities -> Basic: {new_customer_pred[0][0]:.4f} | Pro: {new_customer_pred[0][1]:.4f} | Enterprise: {new_customer_pred[0][2]:.4f}")
if predicted_class == 0:
  print("Assigned Tier Label: BASIC")
elif predicted_class == 1:
  print("Assigned Tier Label: PRO")
else:
  print("Assigned Tier Label: ENTERPRISE")


[TASK 10 SUCCESS] Single-Sample Inference Execution:
Raw Sample Input: [[ 0.95 80.    8.    4.  ]]
Normalized Vector: [[0.8667 1.2078 1.3212 1.069 ]]
Class Probabilities -> Basic: 0.0015 | Pro: 0.0039 | Enterprise: 0.9946
Assigned Tier Label: ENTERPRISE


In [ ]:
print("\n========== WEEK 5 DAY 22: MODULAR DEEP LEARNING ENGINE COMPLETE ==========\n")

print("Architecture Parameters:")
print(f"- Total Network Layers       : {len(network.layers)} Parameterized Layers")
print(f"- Layer Dimensions           : [4, 16, 8, 3]")
print("- Hidden Activation          : ReLU (He Initialized)")
print("- Output Activation          : Softmax (Xavier Initialized)")
print(f"- Dataset Records            : {X_data.shape[0]} Samples (3 Classes)")

print("\nTraining Initialization:")
print("- Loss Function              : Categorical Cross-Entropy (CCE)")
print("- Optimizer                  : Momentum SGD (Beta = 0.9)")
print(f"- Initial Loss (Epoch 1)     : {history[0]:.4f}")
print("- Learning Rate (Alpha)      : 0.1")

print("\nOptimization Progress:")
print(f"- Epoch 30 Loss              : {history[29]:.4f}")
print(f"- Epoch 60 Loss              : {history[59]:.4f}")
print(f"- Epoch 90 Loss              : {history[89]:.4f}")
print(f"- Epoch 120 Loss             : {history[119]:.4f}")

print("\nEvaluation Metrics:")

print(f"- Final Training Loss        : {history[-1]:.4f}")

print(f"- Categorical Accuracy       : {accuracy * 100:.2f}%")

print("- Decision Engine            : Multi-Class Argmax")

print("\nEngine Outcome: Modular N-Layer Engine with Momentum SGD built and verified successfully.")

print("========================================================================")


========== WEEK 5 DAY 22: MODULAR DEEP LEARNING ENGINE COMPLETE ==========

Architecture Parameters:
- Total Network Layers       : 3 Parameterized Layers
- Layer Dimensions           : [4, 16, 8, 3]
- Hidden Activation          : ReLU (He Initialized)
- Output Activation          : Softmax (Xavier Initialized)
- Dataset Records            : 60 Samples (3 Classes)

Training Initialization:
- Loss Function              : Categorical Cross-Entropy (CCE)
- Optimizer                  : Momentum SGD (Beta = 0.9)
- Initial Loss (Epoch 1)     : 1.3264
- Learning Rate (Alpha)      : 0.1

Optimization Progress:
- Epoch 30 Loss              : 0.4766
- Epoch 60 Loss              : 0.1828
- Epoch 90 Loss              : 0.0780
- Epoch 120 Loss             : 0.0330

Evaluation Metrics:
- Final Training Loss        : 0.0330
- Categorical Accuracy       : 100.00%
- Decision Engine            : Multi-Class Argmax

Engine Outcome: Modular N-Layer Engine with Momentum SGD built and verified successfully